# 🛒 Retail Product Sales Prediction
## Course: MIS444 — Predictive Analytics in Business

**Problem Type:** Regression — Predicting a continuous variable (Item Outlet Sales)

**Dataset Source:** [Kaggle — Big Mart Sales Prediction](https://www.kaggle.com/datasets/brijbhushannanda1979/bigmart-sales-data)

---
### Project Outline
1. Data Collection
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Selection
5. Model Building
6. Model Optimization
7. Model Evaluation & Validation
8. Business Insights & Recommendations


---
## 1. Data Collection

The dataset was downloaded from **Kaggle** ([link](https://www.kaggle.com/datasets/brijbhushannanda1979/bigmart-sales-data)). It contains **8,523 product-outlet records** from a retail chain, with **11 predictor features** describing product and outlet characteristics, and the target variable `Item_Outlet_Sales` (sales value in USD).

| Feature | Description |
|---|---|
| Item_Identifier | Unique product ID |
| Item_Weight | Weight of the product |
| Item_Fat_Content | Whether the product is Low Fat or Regular |
| Item_Visibility | % of total display area allocated to this product in the store |
| Item_Type | Category the product belongs to |
| Item_MRP | Maximum Retail Price of the product |
| Outlet_Identifier | Unique store ID |
| Outlet_Establishment_Year | Year the outlet was established |
| Outlet_Size | Size of the outlet (Small/Medium/High) |
| Outlet_Location_Type | Type of city the outlet is located in |
| Outlet_Type | Grocery Store or Supermarket type |
| Item_Outlet_Sales | **Target variable** — sales of the product at that outlet |


In [ ]:
# ── Importing necessary libraries ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, export_graphviz
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("✅ All libraries imported successfully.")


In [ ]:
from google.colab import drive
import os

# Mount drive
drive.mount('/content/drive', force_remount=True)

# Define path and check if file exists to make it robust
file_path = '/content/drive/MyDrive/train.csv'

if os.path.exists(file_path):
    sales = pd.read_csv(file_path)
    sales_original = sales.copy()
    print(f"✅ Dataset loaded. Shape: {sales.shape}")
    display(sales.head(10))
else:
    print(f"❌ File not found at {file_path}. Please check the path.")


---
## 2. Data Preprocessing

Preprocessing ensures the data is clean and ready for modeling. Steps:
- Identify and handle **missing values**
- **Drop irrelevant columns**
- Normalize **inconsistent category labels**
- Check for **invalid values** (e.g. zero visibility)
- **Feature engineer** outlet age from establishment year
- **Encode categorical variables** using One-Hot Encoding
- **Scale** numerical features using StandardScaler


In [ ]:
# ── Step 2.1 : Inspect missing values ──────────────────────────────────────
print("=== Missing Values Per Column ===")
print(sales.isnull().sum())
print(f"\nTotal missing values: {sales.isnull().sum().sum()}")


In [ ]:
# ── Step 2.2 : Visualise missing values ────────────────────────────────────
missing = sales.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x=missing.index, y=missing.values, palette='Reds_r')
plt.title('Missing Value Count by Column', fontsize=14)
plt.xlabel('Column')
plt.ylabel('Missing Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# ── Step 2.3 : Drop irrelevant columns ──────────────────────────────────────
# Item_Identifier is a unique product code, not a measured predictor.
sales.drop(['Item_Identifier'], axis=1, inplace=True)
print(f"Columns after dropping: {sales.shape[1]}")
print(sales.columns.tolist())


In [ ]:
# ── Step 2.3b : Check and remove duplicate rows ─────────────────────────────
duplicates = sales.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

if duplicates > 0:
    sales.drop_duplicates(inplace=True)
    print(f"✅ Duplicates removed. New shape: {sales.shape}")
else:
    print("✅ No duplicate rows — dataset is clean.")


In [ ]:
# ── Step 2.4 : Normalize inconsistent category labels ──────────────────────
# Item_Fat_Content contains inconsistent labels for the same category
# (e.g. 'LF', 'low fat', 'Low Fat' all mean the same thing).
print("Before normalization:", sales['Item_Fat_Content'].unique())

fat_map = {
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'Low Fat': 'Low Fat',
    'reg': 'Regular',
    'Regular': 'Regular'
}
sales['Item_Fat_Content'] = sales['Item_Fat_Content'].map(fat_map).fillna(sales['Item_Fat_Content'])

print("After normalization :", sales['Item_Fat_Content'].unique())


In [ ]:
# ── Step 2.5 : Impute missing values ────────────────────────────────────────
# Item_Weight (numerical) → fill with MEAN (minimises distortion of distribution)
sales['Item_Weight'] = sales['Item_Weight'].fillna(sales['Item_Weight'].mean())

# Outlet_Size (categorical) → fill with MODE (most frequent value)
sales['Outlet_Size'] = sales['Outlet_Size'].fillna(sales['Outlet_Size'].mode()[0])

print("=== Missing Values After Imputation ===")
print(sales.isnull().sum())
print(f"\nTotal remaining missing values: {sales.isnull().sum().sum()}")


In [ ]:
# ── Step 2.6 : Check for invalid values ─────────────────────────────────────
num_cols = ['Item_Weight', 'Item_Visibility', 'Item_MRP']

print("=== Negative Value Check ===")
for col in num_cols + ['Item_Outlet_Sales']:
    has_neg = (sales[col] < 0).any()
    status = "⚠️  HAS negative values" if has_neg else "✅ No negative values"
    print(f"  {col:20s} → {status}")

zero_vis = (sales['Item_Visibility'] == 0).sum()
print(f"\n⚠️  Item_Visibility == 0 for {zero_vis} rows ({zero_vis/len(sales)*100:.2f}%) — a product cannot")
print("   truly have 0% shelf visibility, so these are treated as missing and imputed with the median.")
sales.loc[sales['Item_Visibility'] == 0, 'Item_Visibility'] = sales['Item_Visibility'].median()


In [ ]:
# ── Step 2.7 : Feature engineering — Outlet Age ─────────────────────────────
# Outlet age is more informative for modeling than the raw establishment year.
# Ages are measured relative to the dataset's reference year (2013).
sales['Outlet_Age'] = 2013 - sales['Outlet_Establishment_Year']
sales.drop('Outlet_Establishment_Year', axis=1, inplace=True)
num_cols.append('Outlet_Age')

print(f"✅ Outlet_Age created. Range: {sales['Outlet_Age'].min()}–{sales['Outlet_Age'].max()} years")
sales[['Outlet_Age']].describe().T


In [ ]:
# ── Step 2.8 : Outlier detection & capping (IQR method) ─────────────────────
print("=== Outlier Detection & Capping (IQR Method) ===")
for col in num_cols + ['Item_Outlet_Sales']:
    Q1 = sales[col].quantile(0.25)
    Q3 = sales[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    before = ((sales[col] < lower) | (sales[col] > upper)).sum()
    sales[col] = sales[col].clip(lower=lower, upper=upper)
    print(f"  {col:20s} → {before} outliers found, capped to [{lower:.1f}, {upper:.1f}]  ✅")

print(f"\nTotal rows after capping: {sales.shape[0]}  (no rows dropped)")


In [ ]:
# ── Step 2.9 : One-Hot Encoding for categorical features ────────────────────
cat_cols = ['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier',
            'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']

# Each unique category value becomes a binary (0/1) column.
sales_encoded = pd.get_dummies(sales, columns=cat_cols, dtype=int)

print(f"Shape before encoding : {sales.shape}")
print(f"Shape after encoding  : {sales_encoded.shape}")
sales_encoded.head(3)


# Train/test split

In [ ]:
X_raw = sales_encoded.drop('Item_Outlet_Sales', axis=1)
y     = sales_encoded['Item_Outlet_Sales'].values

# Split BEFORE any scaling
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.3, random_state=42)


Scale only on training data

In [ ]:
# Fit scaler on training data only, then transform both sets
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)   # fit + transform on train
X_test  = scaler.transform(X_test_raw)        # transform only on test

X_scaled = scaler.transform(X_raw)
X        = X_scaled

print(f"✅ Split-then-scale complete. Scaler fitted on training data only.")
print(f"   X_train shape : {X_train.shape}")
print(f"   X_test  shape : {X_test.shape}")
print(f"   y_train shape : {y_train.shape}")
print(f"   y_test  shape : {y_test.shape}")


---
## 3. Exploratory Data Analysis (EDA)

EDA helps us understand the data distribution, relationships between variables, and patterns that may influence the target variable (`Item_Outlet_Sales`).


In [ ]:
# ── EDA 3.1 : Distribution of numerical features ────────────────────────────
sales[num_cols + ['Item_Outlet_Sales']].hist(bins=20, figsize=(16, 10), color='steelblue', edgecolor='white')
plt.suptitle('Distribution of Numerical Features', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()


# Insight:
Item_Outlet_Sales and Item_Visibility are both right-skewed. A log transformation brings
Item_Outlet_Sales closer to normal, which is favorable for linear-type regression models.


In [ ]:
# ── EDA 3.2 : Target variable (Item_Outlet_Sales) distribution ──────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sales['Item_Outlet_Sales'], bins=40, color='coral', edgecolor='white')
axes[0].set_title('Item_Outlet_Sales Distribution (Raw)', fontsize=13)
axes[0].set_xlabel('Item Outlet Sales (USD)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(sales['Item_Outlet_Sales']), bins=40, color='mediumseagreen', edgecolor='white')
axes[1].set_title('Item_Outlet_Sales Distribution (Log-Transformed)', fontsize=13)
axes[1].set_xlabel('log(Item Outlet Sales)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable: Item_Outlet_Sales', fontsize=15)
plt.tight_layout()
plt.show()

print(f"Item_Outlet_Sales — Mean: ${sales['Item_Outlet_Sales'].mean():,.0f}  |  Median: ${sales['Item_Outlet_Sales'].median():,.0f}  |  Std: ${sales['Item_Outlet_Sales'].std():,.0f}")


# Insight:
Item_Outlet_Sales is right-skewed, indicating a long tail of high-selling product/outlet
combinations. After log transformation the distribution is closer to normal.


In [ ]:
# ── EDA 3.3 : Scatter plots — numerical features vs Item_Outlet_Sales ───────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, col in enumerate(num_cols):
    axes[i].scatter(sales[col], sales['Item_Outlet_Sales'], alpha=0.3, color='royalblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Item_Outlet_Sales')
    axes[i].set_title(f'{col} vs Item_Outlet_Sales')

plt.suptitle('Feature vs Item_Outlet_Sales Scatter Plots', fontsize=15)
plt.tight_layout()
plt.show()


# Insight:
Item_MRP shows a clear positive, almost tiered, relationship with sales — higher-priced
items tend to generate higher outlet sales.
# Weakness:
Item_Weight shows very little relationship with sales, suggesting it may not be a strong
standalone predictor.


In [ ]:
# ── EDA 3.4 : Boxplots — Item_Outlet_Sales by categorical features ──────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, col in zip(axes.flatten(), ['Outlet_Type', 'Outlet_Size', 'Outlet_Location_Type', 'Item_Type']):
    order = sales.groupby(col)['Item_Outlet_Sales'].median().sort_values(ascending=False).index
    sns.boxplot(data=sales, x=col, y='Item_Outlet_Sales', order=order, ax=ax, palette='Set2')
    ax.set_title(f'Item_Outlet_Sales by {col}', fontsize=12)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Item_Outlet_Sales Distribution Across Categorical Features', fontsize=15)
plt.tight_layout()
plt.show()


# Insight:
Outlet_Type has the strongest visible impact — Supermarket Type3 outlets sell substantially
more than Grocery Stores, indicating store format is a key driver of sales.


In [ ]:
# ── EDA 3.5 : Correlation heatmap (numerical features) ───────────────────────
numeric_cols = sales[num_cols + ['Item_Outlet_Sales']]

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_cols.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix — Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()


# Insight:
Item_MRP has by far the highest correlation with Item_Outlet_Sales among numerical
features, making it the most important individual numeric predictor.


In [ ]:
# ── EDA 3.6 : Outlet age trend — average sales by outlet age ────────────────
age_sales = sales.groupby('Outlet_Age')['Item_Outlet_Sales'].mean().sort_index()

plt.figure(figsize=(12, 5))
plt.plot(age_sales.index, age_sales.values, color='darkorange', linewidth=1.5, marker='o')
plt.fill_between(age_sales.index, age_sales.values, alpha=0.2, color='darkorange')
plt.title('Average Item_Outlet_Sales by Outlet Age', fontsize=14)
plt.xlabel('Outlet Age (years)')
plt.ylabel('Average Item_Outlet_Sales (USD)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


# Insight:
Average sales vary across outlet ages but do not show a simple linear trend, suggesting
Outlet_Type and Outlet_Size explain more of the variation than outlet age alone.


## 4. Feature Selection

Feature selection identifies which variables most strongly predict `Item_Outlet_Sales`, reducing noise and improving model performance. We use two approaches:
1. **Pearson Correlation** — measures linear relationship between each feature and the target
2. **Random Forest Feature Importance** — captures non-linear relationships as well


In [ ]:
# ── Feature Selection 4.1 : Pearson Correlation with Item_Outlet_Sales ──────
X_df = X_raw.copy()
corr_with_target = X_df.corrwith(pd.Series(y))

top_corr = corr_with_target.abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in corr_with_target[top_corr.index]]
plt.barh(top_corr.index[::-1], top_corr.values[::-1], color=colors[::-1])
plt.title('Top 15 Features by Absolute Correlation with Item_Outlet_Sales', fontsize=13)
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()


In [ ]:
# ── Feature Selection 4.2 : Random Forest Feature Importance ────────────────
X = X_scaled
y = sales_encoded['Item_Outlet_Sales'].values

rf_selector = RandomForestRegressor(n_estimators=100, random_state=42)
rf_selector.fit(X, y)

importances = pd.Series(rf_selector.feature_importances_, index=X_raw.columns)  # ← X_raw.columns, not sales_encoded.columns
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features[::-1].plot(kind='barh', color='teal')
plt.title('Top 15 Features — Random Forest Importance', fontsize=13)
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

print("\nTop 10 important features (Random Forest):")
print(top_features.head(10).round(4))


In [ ]:
# ── Feature Selection 4.3 : Confirm split ────────────────────────────────────
print("=== Train / Test Split (performed in Cell 2.10) ===")
print(f"  Total samples   : {len(X)}")
print(f"  Training samples: {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Testing  samples: {len(X_test)}  ({len(X_test)/len(X)*100:.1f}%)")

overlap = set(X_train_raw.index) & set(X_test_raw.index)
print(f"\nTrain-Test overlap: {'⚠️ OVERLAP FOUND' if overlap else '✅ No overlap — split is clean'}")


In [ ]:
# ── Feature Selection 4.4 : Final feature selection ──────────────────────────

# Combine both methods (Correlation + Random Forest)
common_features = list(set(top_corr.index) & set(top_features.index))

print(f"Selected features ({len(common_features)}):")
print(common_features)

# Create reduced dataset USING selected features
X_selected = X_raw[common_features]

# Train-test split again using selected features
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.3, random_state=42
)

print("\n✅ Feature selection applied successfully")
print(f"New feature count: {X_selected.shape[1]}")


Both Pearson correlation and Random Forest importance were used to identify the most influential features.

The common features from both methods were selected to reduce dimensionality and improve model performance while minimizing noise and overfitting.


---
## 5. Model Building

We train **four regression models** and evaluate them before optimization:

| # | Model | Why chosen |
|---|---|---|
| 1 | Linear Regression | Simple baseline; interpretable coefficients |
| 2 | Decision Tree | Captures non-linear patterns; easy to visualise |
| 3 | Random Forest | Ensemble method; reduces overfitting from single trees |
| 4 | Support Vector Machine (SVM) | Effective in high-dimensional spaces |

> **Metrics used:** MAE, MAPE, MSE, RMSE, R²


In [ ]:
# Use selected features (NOT full dataset)
X = X_selected


In [ ]:
# ── Helper function : compute all regression metrics ────────────────────────
def evaluate_model(y_true, y_pred, model_name):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred) * 100
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

    print(f"\n{'='*45}")
    print(f"  {model_name}")
    print(f"{'='*45}")
    print(f"  MAE  : {mae:>15,.2f}")
    print(f"  MAPE : {mape:>14.2f} %")
    print(f"  MSE  : {mse:>15,.2f}")
    print(f"  RMSE : {rmse:>15,.2f}")
    print(f"  R²   : {r2:>14.4f} %")

    return {'Model': model_name, 'MAE': mae, 'MAPE': mape,
            'MSE': mse, 'RMSE': rmse, 'R2': r2}

results_before = []
print("✅ Helper function ready.")


In [ ]:
# ── Model 5.1 : Linear Regression ────────────────────────────────────────────
linreg = LinearRegression()
linreg.fit(X_train, y_train)

residuals = y_train - linreg.predict(X_train)
mask_clean = np.abs(residuals) <= 3 * np.std(residuals)
linreg.fit(X_train[mask_clean], y_train[mask_clean])

y_pred_lr = linreg.predict(X_test)
res_lr = evaluate_model(y_test, y_pred_lr, 'Linear Regression')
results_before.append(res_lr)

print(f"\n  Train R²: {linreg.score(X_train[mask_clean], y_train[mask_clean])*100:.4f}%")
print(f"  Test  R²: {linreg.score(X_test, y_test)*100:.4f}%")

# Actual vs Predicted plot
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_lr, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Item_Outlet_Sales')
plt.ylabel('Predicted Item_Outlet_Sales')
plt.title('Linear Regression: Actual vs Predicted')
plt.tight_layout()
plt.show()


In [ ]:
# ── Model 5.2 : Decision Tree Regressor ──────────────────────────────────────
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

res_dt = evaluate_model(y_test, y_pred_dt, 'Decision Tree Regressor')
results_before.append(res_dt)

train_r2 = dt.score(X_train, y_train) * 100
test_r2  = dt.score(X_test,  y_test)  * 100

print(f"\n  Train R²: {train_r2:.4f}%")
print(f"  Test  R²: {test_r2:.4f}%")
print(f"\n  ⚠️  NOTE: An unconstrained Decision Tree achieves Train R² ≈ 100%")
print(f"  because it creates a unique leaf for every training sample")
print(f"  (memorization, not learning). Gap = {train_r2 - test_r2:.2f}%.")
print(f"  This overfitting will be corrected in Section 6 via GridSearchCV.")

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_dt, alpha=0.4, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Item_Outlet_Sales')
plt.ylabel('Predicted Item_Outlet_Sales')
plt.title('Decision Tree: Actual vs Predicted (pre-optimization)')
plt.tight_layout()
plt.show()


In [ ]:
# ── Model 5.3 : Random Forest Regressor ──────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

res_rf = evaluate_model(y_test, y_pred_rf, 'Random Forest Regressor')
results_before.append(res_rf)

print(f"\n  Train R²: {rf.score(X_train, y_train)*100:.4f}%")
print(f"  Test  R²: {rf.score(X_test, y_test)*100:.4f}%")

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.4, color='mediumseagreen')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Item_Outlet_Sales')
plt.ylabel('Predicted Item_Outlet_Sales')
plt.title('Random Forest: Actual vs Predicted')
plt.tight_layout()
plt.show()


In [ ]:
# ── Model 5.4 : Support Vector Machine (SVM) ─────────────────────────────────

rob_scaler       = RobustScaler()
X_tr_svm         = rob_scaler.fit_transform(X_train_raw[common_features])
X_te_svm         = rob_scaler.transform(X_test_raw[common_features])

y_tr_svm, y_te_svm = y_train, y_test

svm_model = SVR(kernel='rbf', C=10000, epsilon=0.1)
svm_model.fit(X_tr_svm, y_tr_svm)

y_pred_svm = svm_model.predict(X_te_svm)
res_svm = evaluate_model(y_te_svm, y_pred_svm, 'SVM (SVR)')
results_before.append(res_svm)

train_r2_svm = r2_score(y_tr_svm, svm_model.predict(X_tr_svm)) * 100
test_r2_svm  = r2_score(y_te_svm, y_pred_svm) * 100
print(f"\n  Train R²: {train_r2_svm:.4f}%")
print(f"  Test  R²: {test_r2_svm:.4f}%")
print(f"  Note: SVM uses RobustScaler (different from StandardScaler used by other models).")

plt.figure(figsize=(6, 5))
plt.scatter(y_te_svm, y_pred_svm, alpha=0.4, color='purple')
plt.plot([y_te_svm.min(), y_te_svm.max()], [y_te_svm.min(), y_te_svm.max()], 'r--')
plt.xlabel('Actual Item_Outlet_Sales')
plt.ylabel('Predicted Item_Outlet_Sales')
plt.title('SVM: Actual vs Predicted')
plt.tight_layout()
plt.show()


In [ ]:
results_df = pd.DataFrame(results_before)
results_df = results_df.sort_values(by='R2', ascending=False)

print("\n=== Model Comparison (Before Optimization) ===")
display(results_df)


In [ ]:
best_model = results_df.iloc[0]
print(f"\nBest model before optimization: {best_model['Model']}")


---
## 6. Model Optimization

Optimization improves model performance by tuning **hyperparameters** — settings that are not learned from data but set before training. We use:

- **GridSearchCV** — exhaustively searches through a defined parameter grid
- **5-Fold Cross-Validation** — evaluates each parameter combination on 5 different train/validation splits, reducing the risk of overfitting to a single split

We optimize the two best-performing tree-based models: **Decision Tree** and **Random Forest**.


In [ ]:
# ── Optimization 6.1 : Decision Tree — GridSearchCV ─────────────────────────
print("🔍 Running GridSearchCV for Decision Tree...")

dt_param_grid = {
    'max_depth'        : [3, 5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2', None]
}

dt_grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    dt_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
dt_grid.fit(X_train, y_train)

print(f"\n✅ Best Parameters: {dt_grid.best_params_}")
print(f"   Best CV R²     : {dt_grid.best_score_*100:.4f}%")


In [ ]:
# ── Optimization 6.2 : Decision Tree — evaluate optimized model ─────────────
dt_best = dt_grid.best_estimator_
y_pred_dt_opt = dt_best.predict(X_test)

res_dt_opt = evaluate_model(y_test, y_pred_dt_opt, 'Decision Tree (Optimized)')

print(f"\n  Train R² (optimized): {dt_best.score(X_train, y_train)*100:.4f}%")
print(f"  Test  R² (optimized): {dt_best.score(X_test, y_test)*100:.4f}%")

# Before vs After comparison
print(f"\n{'─'*40}")
print(f"  R² Before Optimization : {res_dt['R2']:.4f}%")
print(f"  R² After  Optimization : {res_dt_opt['R2']:.4f}%")
print(f"  Improvement            : {res_dt_opt['R2'] - res_dt['R2']:+.4f}%")


In [ ]:
# ── Optimization 6.3 : Random Forest — GridSearchCV ─────────────────────────
print("🔍 Running GridSearchCV for Random Forest...")

rf_param_grid = {
    'n_estimators'     : [50, 100, 200],
    'max_depth'        : [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'max_features'     : ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train)

print(f"\n✅ Best Parameters: {rf_grid.best_params_}")
print(f"   Best CV R²     : {rf_grid.best_score_*100:.4f}%")


In [ ]:
# ── Optimization 6.4 : Random Forest — evaluate optimized model ─────────────
rf_best = rf_grid.best_estimator_
y_pred_rf_opt = rf_best.predict(X_test)

res_rf_opt = evaluate_model(y_test, y_pred_rf_opt, 'Random Forest (Optimized)')

print(f"\n  Train R² (optimized): {rf_best.score(X_train, y_train)*100:.4f}%")
print(f"  Test  R² (optimized): {rf_best.score(X_test, y_test)*100:.4f}%")

print(f"\n{'─'*40}")
print(f"  R² Before Optimization : {res_rf['R2']:.4f}%")
print(f"  R² After  Optimization : {res_rf_opt['R2']:.4f}%")
print(f"  Improvement            : {res_rf_opt['R2'] - res_rf['R2']:+.4f}%")


In [ ]:
# ── Optimization 6.5 : Cross-Validation scores (all models) ────────────────
print("=== 5-Fold Cross-Validation (R² scores) ===\n")

from sklearn.pipeline import make_pipeline

for model, name in [(linreg,   'Linear Regression'),
                    (dt_best,  'Decision Tree (Optimized)'),
                    (rf_best,  'Random Forest (Optimized)')]:
    cv_scores = cross_val_score(model, X_selected, y, cv=5, scoring='r2')
    print(f"  {name:30s} → Mean R²: {cv_scores.mean()*100:.4f}%  |  Std: {cv_scores.std()*100:.4f}%")

svm_pipeline = make_pipeline(RobustScaler(), SVR(kernel='rbf', C=10000, epsilon=0.1))
cv_svm = cross_val_score(svm_pipeline, X_raw[common_features], y, cv=5, scoring='r2')
print(f"  {'SVM (SVR)':30s} → Mean R²: {cv_svm.mean()*100:.4f}%  |  Std: {cv_svm.std()*100:.4f}%")
print(f"\n  ✅ SVM CV uses a Pipeline — RobustScaler re-fitted per fold (no leakage).")


---
## 7. Model Evaluation & Validation

We now compare all models side by side using all five regression metrics:

| Metric | Meaning | Lower is better? |
|---|---|---|
| **MAE** | Average absolute prediction error in USD | ✅ |
| **MAPE** | Percentage prediction error | ✅ |
| **MSE** | Mean squared error (penalises large errors more) | ✅ |
| **RMSE** | Square root of MSE — same unit as Item_Outlet_Sales | ✅ |
| **R²** | % of variance in Item_Outlet_Sales explained by the model | ❌ (higher is better) |


In [ ]:
# ── Evaluation 7.1 : Full comparison table ──────────────────────────────────
all_results = [
    res_lr,
    res_dt, res_dt_opt,
    res_rf, res_rf_opt,
    res_svm
]

results_df = pd.DataFrame(all_results).set_index('Model')
results_df = results_df[['MAE', 'MAPE', 'MSE', 'RMSE', 'R2']]
results_df = results_df.round({'MAE': 2, 'MAPE': 4, 'MSE': 2, 'RMSE': 2, 'R2': 4})

print("\n=== Full Model Evaluation Results ===")
print(results_df.to_string())


In [ ]:
# ── Evaluation 7.2 : Visual comparison ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

short_names = ['LinReg', 'DT', 'DT-Opt', 'RF', 'RF-Opt', 'SVM']

axes[0].bar(short_names, results_df['MAE'], color='coral', edgecolor='white')
axes[0].set_title('Mean Absolute Error (MAE)', fontsize=12)
axes[0].set_ylabel('MAE (USD)')
for i, v in enumerate(results_df['MAE']):
    axes[0].text(i, v * 1.01, f'{v:,.0f}', ha='center', fontsize=9)

axes[1].bar(short_names, results_df['MAPE'], color='royalblue', edgecolor='white')
axes[1].set_title('Mean Absolute Percentage Error (MAPE)', fontsize=12)
axes[1].set_ylabel('MAPE (%)')
for i, v in enumerate(results_df['MAPE']):
    axes[1].text(i, v * 1.01, f'{v:.2f}%', ha='center', fontsize=9)

axes[2].bar(short_names, results_df['R2'], color='mediumseagreen', edgecolor='white')
axes[2].set_title('R² Score (%)', fontsize=12)
axes[2].set_ylabel('R² (%)')
for i, v in enumerate(results_df['R2']):
    axes[2].text(i, v * 1.002, f'{v:.2f}%', ha='center', fontsize=9)

for ax in axes:
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Model Comparison — All Metrics', fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
# ── Evaluation 7.3 : Best model — detailed residual analysis ────────────────
# ── Use BEST model (Random Forest, Optimized) for residual analysis ──
residuals_best = y_test - y_pred_rf_opt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals_best, bins=40, color='teal', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Residuals Distribution — Random Forest (Best Model)', fontsize=12)

axes[1].scatter(y_pred_rf_opt, residuals_best, alpha=0.3, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted Values — Random Forest', fontsize=12)

plt.tight_layout()
plt.show()

print(f"Residual Mean : {residuals_best.mean():,.2f}")
print(f"Residual Std  : {residuals_best.std():,.2f}")


### Residual Analysis Insight

The residuals are roughly centered around zero, indicating the model's predictions are not
systematically biased high or low.

The spread widens somewhat for larger predicted sales, suggesting mild heteroscedasticity —
the model is less precise for very high-selling product/outlet combinations.

This confirms the optimized Random Forest captures the main sales patterns well, though
high-value outliers remain the hardest cases to predict exactly.


---
## 8. Business Insights & Recommendations

### 8.1 Key Predictors of Item Outlet Sales

Based on the **Random Forest Feature Importance** analysis and correlation results, the following factors are the most significant drivers of `Item_Outlet_Sales`:

| Rank | Feature | Business Interpretation |
|---|---|---|
| 1 | **Item_MRP** | Maximum retail price — the single strongest predictor. Higher-priced items generate more sales revenue per unit sold. |
| 2 | **Outlet_Type** | Store format — Supermarket Type3 outlets consistently outsell Grocery Stores and smaller supermarket formats. |
| 3 | **Outlet_Identifier / Outlet_Size** | Individual outlet effects — some stores simply perform better, reflecting location, footfall, and size. |
| 4 | **Item_Visibility** | Shelf visibility — products with unusually low or zero visibility tend to sell less. |
| 5 | **Outlet_Age** | Store maturity — established outlets have had more time to build a loyal customer base. |

---
### 8.2 How Businesses Can Apply This Model

**1. Inventory Planning & Replenishment**
Store managers can use predicted sales to plan stock levels for each product-outlet
combination, reducing both stockouts and overstock.

**2. Pricing & Assortment Strategy**
Because Item_MRP is the strongest driver, the retailer can test how modest price
adjustments affect projected sales for specific item categories.

**3. Store Format & Expansion Planning**
Since Outlet_Type strongly influences sales, this informs decisions about which store
formats to prioritise when opening new locations.

**4. Shelf Space Optimization**
Because Item_Visibility affects predicted sales, merchandising teams can reallocate shelf
space toward under-visible but high-potential products.

### 8.3 Model Selection Summary

The **Random Forest (Optimized)** model is selected as the best model for deployment because:

- It achieved the highest R² score among all models tested, explaining the most variance in Item_Outlet_Sales
- It has a low RMSE relative to the baseline models
- It handles the mix of numerical and one-hot encoded categorical features well without requiring feature scaling
- Cross-validation confirms consistent performance across different data splits

---
### 8.4 Limitations & Future Improvements

| Limitation | Suggested Improvement |
|---|---|
| Dataset is a single snapshot in time | Collect multi-year sales data to capture seasonality and trends |
| Outlet_Establishment_Year is only available at the store level | Add outlet-level foot-traffic or regional economic data |
| Only 11 predictor features available | Incorporate promotions, competitor pricing, or supply-chain data |
| Item_Visibility of 0 required imputation, adding some noise | Investigate the data-collection process behind visibility measurement |
| Very high-sales outliers are the hardest cases to predict exactly | Train a separate model or use quantile regression for top-tier sales |


In [ ]:
# ── Insights 8.1 : Outlet type sales analysis ────────────────────────────────
outlet_type_sales = sales.groupby('Outlet_Type')['Item_Outlet_Sales'].agg(['mean', 'count']).reset_index()
outlet_type_sales.columns = ['Outlet_Type', 'AvgSales', 'Count']
outlet_type_sales = outlet_type_sales.sort_values('AvgSales', ascending=False)

plt.figure(figsize=(10, 5))
bars = plt.bar(outlet_type_sales['Outlet_Type'],
               outlet_type_sales['AvgSales'],
               color=plt.cm.RdYlGn(outlet_type_sales['AvgSales'] /
                                   outlet_type_sales['AvgSales'].max()))
plt.xticks(rotation=20, ha='right')
plt.title('Average Item_Outlet_Sales by Outlet Type', fontsize=14)
plt.xlabel('Outlet Type')
plt.ylabel('Average Item_Outlet_Sales (USD)')
plt.axhline(sales['Item_Outlet_Sales'].mean(), color='navy', linestyle='--',
            label=f'Overall Mean: ${sales["Item_Outlet_Sales"].mean():,.0f}')
plt.legend()
plt.tight_layout()
plt.show()


### Insight: Outlet Type Impact

The chart shows that Supermarket Type3 outlets generate substantially higher average
sales than Grocery Stores and other supermarket formats.

👉 Business Use: When planning new store openings or renovations, the retailer should
prioritize the Supermarket Type3 format where feasible to maximize expected sales.


In [ ]:
# ── Insights 8.2 : Feature impact simulation (Item_MRP) ──────────────────────
mrp_range = np.arange(sales['Item_MRP'].min(), sales['Item_MRP'].max(), 5)
base_row = pd.DataFrame([X_raw.iloc[0].copy()])  # ← X_raw, not sales_encoded

simulated_sales = []
for mrp in mrp_range:
    row = base_row[common_features].copy()
    row['Item_MRP'] = mrp
    simulated_sales.append(svm_model.predict(rob_scaler.transform(row))[0])

plt.figure(figsize=(10, 5))
plt.plot(mrp_range, simulated_sales, color='steelblue', linewidth=2, marker='o', markersize=3)
plt.fill_between(mrp_range, simulated_sales, alpha=0.15, color='steelblue')
plt.title('Simulated Impact of Item_MRP on Predicted Item_Outlet_Sales', fontsize=13)
plt.xlabel('Item_MRP (USD)')
plt.ylabel('Predicted Item_Outlet_Sales (USD)')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("Business Insight: Each $5 increase in Item_MRP is associated with")
print(f"an estimated sales change of approx. ${(simulated_sales[-1]-simulated_sales[0])/(len(mrp_range)-1):,.2f} per unit of MRP step.")


Item_MRP: Higher-priced items are associated with higher outlet sales
Business Use: Category managers can use this relationship to evaluate the sales impact
of repositioning products into higher price tiers.


In [ ]:
# ── Final Summary ────────────────────────────────────────────────────────────
best_model = results_df.loc[results_df['R2'].idxmax()]
print("Final model selected based on highest R² and lowest error metrics\n")
print("╔══════════════════════════════════════════════════════════╗")
print("║      MIS444 — Retail Product Sales Prediction Summary     ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Dataset          : 8,523 records, 11 features            ║")
print(f"║  Problem Type     : Regression                            ║")
print(f"║  Models Trained   : 4 (LR, DT, RF, SVM)                   ║")
print(f"║  Models Optimized : 2 (DT, RF via GridSearchCV)           ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Best Model       : {best_model.name:<38s}║")
print(f"║  Best R²          : {best_model['R2']:.4f}%{' '*30}║")
print(f"║  Best MAE         : ${best_model['MAE']:>10,.2f} USD{' '*22}║")
print(f"║  Best MAPE        : {best_model['MAPE']:.4f}%{' '*30}║")
print(f"║  Best RMSE        :${best_model['RMSE']:>10,.2f} USD{' '*22}║")
print("╚══════════════════════════════════════════════════════════╝")


### Final Conclusion

This project applied predictive analytics techniques to estimate retail product sales
using multiple regression models on the Big Mart Sales dataset.

Among all models, the optimized **Random Forest** performed the best, achieving the
highest accuracy and lowest prediction error after hyperparameter tuning.

The model can be used by retail managers, category planners, and inventory teams to
support stocking, pricing, and store-format decisions.

Future improvements include incorporating promotions, competitor pricing, and multi-year
sales history to improve generalizability and capture seasonal effects.
